# Healthcare-App Clinical Knowledge Assistant
## Track 2 — Hybrid RAG with BM25 + Semantic + ReAct Agent

This notebook walks through the full system in seven sections:
- **Section 0** — Setup & Architecture
- **Section 1** — Vector Store Initialisation
- **Section 2** — BM25 Keyword Retriever
- **Section 3** — Semantic Retriever
- **Section 4** — Hybrid Fusion (RRF)
- **Section 5** — PDF Ingestion Demo
- **Section 6** — Comparison Dashboard
- **Section 7** — ReAct Agent with Nebius AI Studio

---
## Section 0 — Setup & Architecture

In [ ]:
# Install dependencies (run once)
# !pip install langchain langchain-openai langchain-pinecone langchain-community \
#   langchain-chroma langchain-huggingface langchain-text-splitters \
#   pinecone-client chromadb sentence-transformers rank-bm25 \
#   pymupdf pdfplumber altair pandas matplotlib streamlit python-dotenv fpdf2 openai numpy

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path

load_dotenv()

ARCH = """
┌─────────────────────────────────────────────────────────────────────────────┐
│                   Healthcare-App Clinical Knowledge Assistant               │
│                                                                             │
│  Care Coordinator Query (natural language + MRN / bug ID)                  │
│              │                                                              │
│              ▼                                                              │
│  ┌─────────────────────────────────────────────────────────┐               │
│  │       ReAct Agent  (build_llm() → Nebius/OpenAI)       │               │
│  │  Tool 1: search_kb_semantic  — vector cosine similarity │               │
│  │  Tool 2: search_kb_bm25      — BM25 keyword scoring     │               │
│  │  Tool 3: search_kb_hybrid    — RRF fusion               │               │
│  │  Tool 4: lookup_patient_record — mock Epic FHIR API     │               │
│  │  Tool 5: check_system_status   — mock incident API      │               │
│  └─────────────────────────────────────────────────────────┘               │
│              │                                                              │
│    ┌─────────┴──────────┐                                                  │
│    ▼                    ▼                                                   │
│  Pinecone (primary)   ChromaDB (fallback)                                  │
│  512-dim cosine       384-dim cosine                                       │
│  text-embedding-3-small   all-MiniLM-L6-v2                                 │
│    ▲                                                                        │
│  PDF Ingestor (PyMuPDF) — rebuilds BM25 after upload                      │
└─────────────────────────────────────────────────────────────────────────────┘
"""
print(ARCH)

store_type = "Pinecone" if os.getenv("PINECONE_API_KEY") else "ChromaDB (fallback)"
print(f"Active vector store: {store_type}")

In [ ]:
import json
KB_PATH = Path("healthcare_app_knowledge_base.json")
docs = json.loads(KB_PATH.read_text())
print(f"Knowledge base loaded: {len(docs)} documents")
from collections import Counter
print("By doc_type:", dict(Counter(d['doc_type'] for d in docs)))

✅ **What you learned in Section 0:** The system has two execution modes (Pinecone cloud / ChromaDB local) controlled purely by environment variables. The ReAct agent sits above the retrieval layer and autonomously selects which of its five tools to call.

---
## Section 1 — Vector Store Initialisation

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s — %(message)s")

from track2_retrieval_engine import RetrievalEngine

engine = RetrievalEngine(KB_PATH)
print(f"\nVector store type: {engine._store_type}")
print(f"Chunks in BM25 index: {len(engine._all_chunks)}")

### 🧠 Teaching note — why 512 dims and cosine?

**`text-embedding-3-small` at 512 dims** is a good trade-off for clinical text:
- Smaller than 1536 dims → faster indexing, lower Pinecone cost
- Larger than 384 dims (MiniLM) → better representation for medical vocabulary

**Cosine metric** is preferred over L2 for text because cosine ignores document length — a long runbook and a short FAQ can still score close to 1.0 if they discuss the same concept.

**ChromaDB fallback uses MiniLM (384 dims)** — free, local, no API key needed. Great for development but lower accuracy on domain-specific clinical queries than OpenAI embeddings.

**What happens to rare medical identifiers like `ERR_HL7_ROUTE_FAIL`?**  
Embeddings smear meaning across dimensions, so a rare token like `ERR_HL7_ROUTE_FAIL` is represented as a vector similar to other error-code-like tokens. The semantic retriever might not return an exact match. This is exactly where BM25 excels — see Section 2.

✅ **What you learned in Section 1:** The engine auto-selects Pinecone or ChromaDB based on environment variables, chunks the KB into 500-token overlapping windows, and skips re-indexing if the vector store already has data.

---
## Section 2 — BM25 Keyword Retriever

In [ ]:
# Three canonical queries
QUERIES = [
    "patient can't book follow-up after discharge — conflict error",   # conceptual
    "ERR_HL7_ROUTE_FAIL lab result not in chart",                       # exact identifier
    "MRN-334521 duplicate metoprolol 25mg MAR BUG-EHR-2301",           # mixed
]

print("=" * 64)
print("BM25 RESULTS")
print("=" * 64)
for q in QUERIES:
    print(f"\nQuery: {q}")
    results = engine.search_bm25(q, k=5)
    for r in results:
        print(f"  [{r.score:7.3f}] {r.title}  ({r.doc_type}/{r.clinical_area})")

### 🧠 Teaching note — IDF weighting in BM25

BM25 uses **Inverse Document Frequency (IDF)**: rare tokens get high scores, common tokens get low scores.

| Token | Frequency in KB | IDF impact |
|-------|----------------|------------|
| `ERR_HL7_ROUTE_FAIL` | appears in ~5 docs | **high IDF → high BM25 score** |
| `BUG-EHR-2301` | appears in ~6 docs | **high IDF → high BM25 score** |
| `patient` | appears in almost every doc | **near-zero IDF → low contribution** |
| `MRN-334521` | appears in ~3 docs | **very high IDF → highest BM25 score** |

This is why BM25 is the right tool for queries containing exact identifiers — the rarity of those tokens in the corpus makes them the dominant signal.

✅ **What you learned in Section 2:** BM25 rewards rare, exact tokens. Error codes and MRN identifiers produce high BM25 scores. The word "patient" contributes almost nothing. This is the opposite of semantic search.

---
## Section 3 — Semantic Retriever

In [ ]:
print("=" * 64)
print("SEMANTIC RESULTS (cosine similarity)")
print("=" * 64)
for q in QUERIES:
    print(f"\nQuery: {q}")
    results = engine.search_semantic(q, k=5)
    for r in results:
        print(f"  [{r.score:.4f}] {r.title}  ({r.doc_type}/{r.clinical_area})")

### 🧠 Teaching note — synonym bridging and identifier noise

**Synonym bridging:** Notice that Query 1 uses the phrase "conflict error" — semantic search finds documents about `SCH-CONFLICT-88` and scheduling problems even without the exact error code. The embedding model understands that "conflict error" ≈ "booking conflict" ≈ `SCH-CONFLICT-88`.

**Why exact IDs are noisy for semantic search:** `ERR_HL7_ROUTE_FAIL` is an unusual token sequence. The embedding model has rarely seen it, so it maps it to a generic 'error token' region of the embedding space rather than the HL7-specific region. This produces lower-quality results compared to BM25.

**The key insight:** Semantic excels at intent matching across paraphrase boundaries. BM25 excels at exact token matching. Neither alone is optimal — you need both.

✅ **What you learned in Section 3:** Semantic search bridges vocabulary gaps ("conflict error" → scheduling bug docs) but is noisier for exact identifiers. Cosine scores above 0.80 indicate strong semantic similarity.

---
## Section 4 — Hybrid Fusion (RRF)

In [ ]:
def show_hybrid(query: str, k: int = 5) -> None:
    """Print VECTOR / BM25 / ENSEMBLE results side by side."""
    sem = engine.search_semantic(query, k=k)
    bm25 = engine.search_bm25(query, k=k)
    hybrid = engine.search_hybrid(query, k=k)

    print(f"\nQuery: {query}")
    print(f"{'VECTOR (cosine)':<48} {'BM25':^20} {'HYBRID (RRF)':>20}")
    print("-" * 92)
    for i in range(max(len(sem), len(bm25), len(hybrid))):
        s = sem[i].title[:44] if i < len(sem) else ""
        b = bm25[i].title[:18] if i < len(bm25) else ""
        h = hybrid[i].title[:18] if i < len(hybrid) else ""
        tag = f"[{hybrid[i].retriever}]" if i < len(hybrid) else ""
        print(f"  {s:<46} | {b:<20} | {h:<20} {tag}")

for q in QUERIES:
    show_hybrid(q)
    print()

In [ ]:
# RAG answer for Query 3 using build_llm()
from track2_retrieval_engine import build_llm

q3 = QUERIES[2]  # mixed query with MRN + bug ID
context_docs = engine.search_hybrid(q3, k=5)
context = "\n\n---\n\n".join(f"**{r.title}**\n{r.content[:300]}" for r in context_docs)

print("Context documents retrieved:")
for r in context_docs:
    print(f"  [{r.retriever}] vrank={r.vector_rank} brank={r.bm25_rank}  {r.title}")

RAG_PROMPT = """You are a clinical operations assistant. Answer using only the context below.
Be concise: state the immediate step and cite sources.

Context: {context}
Question: {question}
Answer:"""

try:
    from langchain_core.prompts import PromptTemplate
    llm = build_llm()
    chain = PromptTemplate.from_template(RAG_PROMPT) | llm
    response = chain.invoke({"context": context, "question": q3})
    print("\n📋 RAG Answer:")
    print(response.content if hasattr(response, 'content') else str(response))
except Exception as e:
    print(f"⚠️  LLM not available ({e}). Set OPENAI_API_KEY in .env to see the generated answer.")

### 🧠 Teaching note — RRF formula and weight trade-off

**Reciprocal Rank Fusion formula:**
```
score(doc) = vector_weight / (RRF_K + vector_rank) + bm25_weight / (RRF_K + bm25_rank)
```
where `RRF_K = 60` (standard constant that dampens the importance of top-ranked results).

**Why RRF_K = 60?** It means rank 1 contributes `1/61 ≈ 0.016` and rank 10 contributes `1/70 ≈ 0.014`. The difference between rank 1 and rank 10 is small (~15%), which makes RRF robust to noise in individual rankers.

**Weight trade-off (default: vector=0.6, BM25=0.4):**
- Higher vector weight → better for conceptual queries without exact identifiers
- Higher BM25 weight → better for queries with MRNs, error codes, bug IDs
- `hybrid-both` tag means the document was retrieved by both — these are the highest-confidence results

✅ **What you learned in Section 4:** RRF fuses rankings without requiring comparable score scales. Documents retrieved by both rankers (`hybrid-both`) are the most trustworthy results. The 0.6/0.4 default weight favours semantic but still rewards exact identifier matches from BM25.

---
## Section 5 — PDF Ingestion Demo

In [ ]:
# Create a synthetic HIPAA Retention Policy PDF for demo
try:
    from fpdf import FPDF
except ImportError:
    import subprocess; subprocess.run(["pip", "install", "-q", "fpdf2"])
    from fpdf import FPDF

import tempfile

PDF_CONTENT = """HIPAA Data Retention Policy — Healthcare-App
Version 1.0 | Effective: 2024-01-01

1. Scope
This policy applies to all electronic protected health information (ePHI) processed
by Healthcare-App, including patient records, lab results, telehealth recordings,
and medication administration records (MAR).

2. Retention Periods
- Adult patient records: 10 years from last encounter
- Telehealth session recordings: 7 years
- MAR and medication orders: 5 years
- Lab results (HL7 ORU^R01 messages): 7 years
- Audit logs (system access, EHR changes): 6 years

3. HIPAA Compliance Notes
All data must be stored using AES-256 encryption at rest.
Access to ePHI requires MFA. Patient data must never leave the hospital
network without a signed BAA. Nebius AI Studio must be used for all AI inference
queries containing real MRN identifiers or other PHI.

4. Disposal
Records past retention must be destroyed using DoD 5220.22-M standard.
Destruction must be logged with date, method, and authorising officer.
"""

pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=11)
for line in PDF_CONTENT.split("\n"):
    pdf.multi_cell(0, 6, line)

tmp_pdf = Path(tempfile.mktemp(suffix="_hipaa_retention.pdf"))
pdf.output(str(tmp_pdf))
print(f"Created synthetic PDF: {tmp_pdf.name} ({tmp_pdf.stat().st_size} bytes)")

In [ ]:
from track2_pdf_ingestor import PDFIngestor

ingestor = PDFIngestor(engine._vector_store)
n_chunks = ingestor.ingest_file(
    tmp_pdf,
    doc_type="product_doc",
    clinical_area="account",
    priority="P1",
    patient_tier="all",
)
print(f"Ingested {n_chunks} chunks from the HIPAA Retention Policy PDF.")

# Rebuild BM25 so the new content is keyword-searchable
engine.rebuild_bm25()
print("BM25 index rebuilt.")

In [ ]:
# Verify the new content is searchable
print("Semantic search — 'HIPAA data retention ePHI telehealth':")
for r in engine.search_semantic("HIPAA data retention ePHI telehealth", k=3):
    src = f" (📄 {r.source_file} p.{r.page_number})" if r.source_file else ""
    print(f"  [{r.score:.4f}] {r.title}{src}")

print("\nBM25 search — 'MAR retention years AES-256':")
for r in engine.search_bm25("MAR retention years AES-256", k=3):
    src = f" (📄 {r.source_file} p.{r.page_number})" if r.source_file else ""
    print(f"  [{r.score:.4f}] {r.title}{src}")

tmp_pdf.unlink(missing_ok=True)  # clean up temp file

### 🧠 Teaching note — PDF loaders, PHI risk, and page_number citation

**PyMuPDF vs pdfplumber vs unstructured:**
| Library | Speed | Tables | Accuracy | Use case |
|---------|-------|--------|----------|----------|
| PyMuPDF | Fastest | Basic | High | Clinical PDFs, dense text |
| pdfplumber | Medium | Excellent | High | PDFs with structured tables |
| unstructured | Slowest | Best | Highest | Mixed-format docs (images, forms) |

**PHI warning:** Manual PDF upload is the highest-risk ingestion path — a user could accidentally upload a document containing real MRNs. The `page_number` metadata is critical for audit trails: if a piece of guidance is later found to be incorrect, you can trace it back to the exact page of the exact document that was ingested.

**Production recommendation:** All PDF ingestion in production should require Nebius AI Studio mode to be active, and should run through an automated PHI-detection scan before embedding.

✅ **What you learned in Section 5:** PDFs are extracted page-by-page, cleaned of headers/footers, chunked, and upserted into the same vector store as the JSON KB. BM25 must be explicitly rebuilt after ingestion. The `source_file` and `page_number` metadata enable auditability.

---
## Section 6 — Comparison Dashboard

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

Q3 = QUERIES[2]  # mixed query — most interesting for comparison

rows = []
for mode, results in [
    ("Semantic", engine.search_semantic(Q3, k=5)),
    ("BM25",     engine.search_bm25(Q3, k=5)),
    ("Hybrid",   engine.search_hybrid(Q3, k=5)),
]:
    for rank, r in enumerate(results):
        rows.append({"Mode": mode, "Rank": rank + 1, "Title": r.title[:35], "Score": r.score or 0.0})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {"Semantic": "#0d6e8a", "BM25": "#e67e22", "Hybrid": "#27ae60"}

for ax, mode in zip(axes, ["Semantic", "BM25", "Hybrid"]):
    sub = df[df["Mode"] == mode].sort_values("Score", ascending=True)
    ax.barh(sub["Title"], sub["Score"], color=colors[mode], edgecolor="white")
    ax.set_title(f"{mode} Search", fontsize=13, fontweight="bold", color=colors[mode])
    ax.set_xlabel("Score")
    ax.tick_params(axis='y', labelsize=8)

fig.suptitle(f"Query 3: '{Q3[:55]}…'\nRetrieval comparison across three modes", fontsize=11)
plt.tight_layout()
plt.savefig("retrieval_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Chart saved to retrieval_comparison.png")

✅ **What you learned in Section 6:** The three modes return overlapping but distinct result sets. Semantic scores and BM25 scores are not directly comparable — they live on different scales. RRF normalises everything to rank-based fusion scores in `[0, 0.02]`, making them comparable.

---
## Section 7 — ReAct Agent with Nebius AI Studio

### The ReAct Loop

```
User Query
    │
    ▼
Thought: "What do I need to answer this?"
    │
    ▼
Action: pick_tool(tool_name, tool_input)
    │
    ▼
Observation: tool_result
    │
    ▼
Thought: "Do I have enough info?"  ──────────┐
    │                                        │ no → repeat
    ▼ yes                                    │
Final Answer (cited, multi-source)  ◄────────┘
```

### Nebius AI Studio vs OpenAI Cloud — `build_llm()` factory

```python
# Development (non-PHI queries)
OPENAI_API_KEY=sk-...        # → ChatOpenAI(model="gpt-4.1-mini")

# Production (PHI-containing queries — Nebius required)
NEBIUS_API_KEY=...           # → ChatOpenAI pointed at Nebius AI Studio
NEBIUS_MODEL_NAME=meta-llama/Meta-Llama-3.1-70B-Instruct
# API base: https://api.studio.nebius.com/v1
```
No code change — only environment variables switch.

In [ ]:
import os
mode = "Nebius AI Studio (PHI-safe)" if os.getenv("NEBIUS_API_KEY") else "OpenAI Cloud (development only)"
print(f"LLM backend: {mode}")
if not (os.getenv("OPENAI_API_KEY") or os.getenv("NEBIUS_API_KEY")):
    print("⚠️  No API key set. Set OPENAI_API_KEY or NEBIUS_API_KEY in .env to run the agent.")

In [ ]:
from track2_clinical_agent import ClinicalAgent

def run_case(label: str, query: str) -> None:
    print(f"\n{'='*64}")
    print(f"Case {label}")
    print(f"Query: {query}")
    print("=" * 64)

    if not (os.getenv("OPENAI_API_KEY") or os.getenv("NEBIUS_API_KEY")):
        print("⚠️  Skipped — no API key. Set OPENAI_API_KEY or NEBIUS_API_KEY in .env")
        return

    agent = ClinicalAgent(engine)
    result = agent.run(query)

    print("\n🔧 Tool calls (intermediate steps):")
    for action, observation in result["intermediate_steps"]:
        obs_preview = str(observation)[:200].replace("\n", " ")
        print(f"  → {action.tool}({action.tool_input!s:.80})")
        print(f"    ↳ {obs_preview}…")

    print(f"\n💬 Final answer:\n{result['output']}")

In [ ]:
run_case(
    "1 — Identifier only",
    "Is BUG-EHR-2301 still active and what's the workaround?",
)

In [ ]:
run_case(
    "2 — Conceptual multi-step",
    "A patient discharged yesterday can't book a follow-up, getting a conflict error — "
    "what's happening and when will it be fixed?",
)

In [ ]:
run_case(
    "3 — Full pipeline (qualifying scenario — all 5 tools)",
    "MRN-334521 has two metoprolol 25mg orders at 08:00. Patient hasn't received either "
    "dose yet. What's the immediate step and is BUG-EHR-2301 still active?",
)

### 🧠 Teaching note — ReAct loop mechanics and Nebius as a production requirement

**Why tool docstring quality matters:** The agent decides which tool to call based *entirely* on the docstring. Compare:
- Bad: `"Search the knowledge base"` → agent can't distinguish semantic vs BM25 vs hybrid
- Good: `"Use when the query contains exact identifiers: MRN-XXXXXX, ERR_*, BUG-*"` → agent reliably picks BM25

**`max_iterations=6` safety:** Without this, a confused agent can loop indefinitely. The `early_stopping_method="generate"` forces a best-effort final answer when the limit is hit.

**Why Nebius AI Studio is required for Case 3:**  
Case 3 query contains `MRN-334521` — a real patient identifier (PHI). If only `OPENAI_API_KEY` is active, that MRN is transmitted to OpenAI's servers, violating HIPAA.
With `NEBIUS_API_KEY` set, the query is processed via Nebius AI Studio (an OpenAI-compatible endpoint you control).
`build_llm()` switches automatically — no code change, just environment variables.

**Production rule:** Any query that might contain a real MRN, real name, or other PHI must only be processed when `NEBIUS_API_KEY` is set. The AI Agent tab in the Streamlit app shows a PHI warning when it detects an MRN pattern in the query.

✅ **What you learned in Section 7:** The ReAct agent autonomously sequences tool calls based on docstring guidance. Nebius AI Studio is the production-safe backend for queries containing PHI — `build_llm()` handles the switch transparently via environment variables. The qualifying Case 3 scenario exercises all five tools in one query.

---
## Summary

| Component | File | Status |
|-----------|------|--------|
| Knowledge Base (50 docs) | `healthcare_app_knowledge_base.json` | ✅ |
| Retrieval Engine (3 modes) | `track2_retrieval_engine.py` | ✅ |
| PDF Ingestor | `track2_pdf_ingestor.py` | ✅ |
| Clinical ReAct Agent | `track2_clinical_agent.py` | ✅ |
| Streamlit KB Viewer (5 tabs) | `track2_kb_viewer.py` | ✅ |
| This notebook | `track2_hybrid_rag.ipynb` | ✅ |

### To run the app:
```bash
# 1. Set credentials in .env (copy from .env.example)
# 2. Launch the Streamlit app
streamlit run track2_kb_viewer.py

# For Nebius mode (production / PHI-safe):
# Add to .env:
#   NEBIUS_API_KEY=...
#   NEBIUS_MODEL_NAME=meta-llama/Meta-Llama-3.1-70B-Instruct
# build_llm() auto-activates Nebius — no code change needed
```